# 08.06 · Agentic RAG — Milestone M3

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Turn retrieval into a **tool** the model decides when and how to call
- [ ] Let the model search iteratively (refine queries, search again for missing pieces)
- [ ] Compare with single-shot RAG on multi-part questions, including cost

## Resources
- [OpenAI – Function calling](https://platform.openai.com/docs/guides/function-calling) — free
- [DeepLearning.AI – Building Agentic RAG with LlamaIndex (short course)](https://www.deeplearning.ai/short-courses/building-agentic-rag-with-llamaindex/) — free
- [LangGraph – Agentic RAG tutorial](https://langchain-ai.github.io/langgraph/tutorials/rag/langgraph_agentic_rag/) — free

Single-shot RAG retrieves once with the raw question. Questions like *"Compare when Lincoln was born and when he became president"*
need two separate searches. An agent can do that — chapter 09 generalises the loop you write here.

In [ ]:
import json
import re

from aieng import rag
from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. The tool
A JSON-schema description the model reads, plus the Python function we run when it asks.

In [ ]:
TOOLS = [{
    "type": "function",
    "function": {
        "name": "search_docs",
        "description": "Search an encyclopedia corpus. Returns the 4 most relevant passages with ids. "
                       "Call it several times with focused queries for questions with several parts.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "a focused search query"}}, "required": ["query"]},
    },
}]


def search_docs(query):
    return "\n\n".join(f"[{pid}] {text}" for pid, _, text in rag.search(query, k=4))

## 2. The loop
Ask → if the model returns tool calls, run them, append results as `tool` messages, ask again → until it answers (or we hit a step limit).
`tool_choice`: `"auto"` (model decides), `"required"` (must call some tool), `"none"`, or a specific function. Forcing the first
search is a cheap fix for models that answer from memory — or give up — without looking. Ollama ignores `tool_choice`, so the loop
also nudges ("Use the tools before answering.") when step 0 returns no tool call.

In [ ]:
SYSTEM = ("You answer questions using the search_docs tool. Search before answering; search again if information is missing. "
          "Answer concisely, cite passage ids like [12], and say \"I don't know\" if the corpus doesn't contain the answer.")


def agentic_answer(question, max_steps=5, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    searches, tokens = [], 0
    for step in range(max_steps):
        # Force a search on step 0 (small models sometimes answer "I don't know" without looking); let it decide afterwards.
        r = c.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0, max_tokens=800,
                                      tool_choice="required" if step == 0 else "auto", **EXTRA)
        msg = r.choices[0].message
        tokens += r.usage.total_tokens
        if not msg.tool_calls and step == 0:  # Ollama ignores tool_choice: nudge and retry instead
            messages.append({"role": "user", "content": "Use the tools before answering."})
            continue
        if not msg.tool_calls:
            return msg.content, searches, tokens
        messages.append(msg.model_dump(exclude_none=True))
        for call in msg.tool_calls:
            query = json.loads(call.function.arguments).get("query", "")
            searches.append(query)
            if verbose:
                print(f"  🔎 search_docs({query!r})")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": search_docs(query)})
    return "Stopped: step limit reached.", searches, tokens


text, searches, tokens = agentic_answer("How old was Lincoln when he became president, and in what year was he born?")
print(text, f"\n[{len(searches)} searches, {tokens} tokens]")

## 3. Single-shot vs agentic on multi-part questions

In [ ]:
QUESTIONS = [  # (question, list of strings a good answer must contain)
    ("In what year was Lincoln born and in what year was he assassinated?", ["1809", "1865"]),
    ("What is the capital of Uruguay, and what language is spoken there?", ["Montevideo", "Spanish"]),
    ("Which is longer-lived on average, elephants or kangaroos? Give rough lifespans.", ["elephant"]),
    ("Who was Lincoln's wife and how many sons did they have?", ["Mary", "four"]),
    ("What is the Gettysburg Address and when was it delivered?", ["1863"]),
]


def single_shot(question):
    text, hits = rag.answer(question, k=4)
    return text


def score(text, must):
    return all(m.lower() in text.lower() for m in must)


rows = []
for q, must in QUESTIONS:
    s = single_shot(q)
    a, searches, tokens = agentic_answer(q, verbose=False)
    rows.append((score(s, must), score(a, must), len(searches)))
    print(f"{q}\n  single: {'✓' if rows[-1][0] else '✗'} {s[:110]!r}\n  agent : {'✓' if rows[-1][1] else '✗'} ({len(searches)} searches) {a[:110]!r}\n")

print(f"single-shot {sum(r[0] for r in rows)}/{len(rows)} | agentic {sum(r[1] for r in rows)}/{len(rows)} | avg searches {sum(r[2] for r in rows) / len(rows):.1f}")

Trade-offs: agentic RAG costs more calls, more tokens and more latency, and small models sometimes skip the tool or loop. Use it when questions
are genuinely multi-step; use single-shot (fast, predictable) for the common case — or route between them (09.03).

## Milestone M3 — deliverable
A Q&A app over **your own documents** with:
- [ ] parsing + structure-aware chunking (08.02)
- [ ] hybrid search + reranking (07.04–07.05)
- [ ] citations and "I don't know" (08.01)
- [ ] a 30+ question eval set with measured retrieval recall@5 and answer accuracy (chapter 10 makes it rigorous)
- [ ] optional: agentic mode for multi-part questions (this lesson)

## Exercises
1. Add a second tool, `get_passage(id)`, that returns one passage by id, and a `neighbors(id)` tool returning adjacent passages. Does the agent use them?
2. Run the comparison with a hosted model (`client("groq")`, `client("gemini")`). How do search counts and accuracy change?
3. Add a hard budget: stop after N total tokens and return the best partial answer.

In [ ]:
# your code here